In [1]:
!pip install pandas==2.1.0 boto3==1.28.0 python-dotenv==1.0.0 duckdb==0.10.0 pyarrow==14.0.1 slack-sdk==3.27.1

In [2]:
!pip install apache-airflow==2.8.1 --constraint "https://raw.githubusercontent.com/apache/airflow/constraints-2.8.1/constraints-3.11.txt"

In [3]:
import pandas as pd
import pyarrow
import duckdb
import boto3
import importlib.metadata

print("✅ pandas     :", pd.__version__)
print("✅ pyarrow    :", pyarrow.__version__)
print("✅ duckdb     :", duckdb.__version__)
print("✅ boto3      :", boto3.__version__)
print("✅ slack_sdk :", importlib.metadata.version("slack_sdk"))
print("\n🎉 All libraries loaded successfully!")

✅ pandas     : 3.0.1
✅ pyarrow    : 23.0.1
✅ duckdb     : 1.5.1
✅ boto3      : 1.28.0
✅ slack_sdk : 3.27.1

🎉 All libraries loaded successfully!


In [7]:
!pip install --force-reinstall --no-cache-dir pandas pyarrow duckdb

   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.9 MB ? eta -:--:--
   -- ------------------------------------- 0.5/9.9 MB 1.0 MB/s eta 0:00:09
   --- ------------------------------------ 0.8/9.9 MB 1.0 MB/s eta 0:00:09
   ----- ---------------------------------- 1.3/9.9 MB 1.9 MB/s eta 0:00:05
   -------- ------------------------------- 2.1/9.9 MB 2.3 MB/s eta 0:00:04
   ------------- -------------------------- 3.4/9.9 MB 2.9 MB/s eta 0:00:03
   ------------------- -------------------- 4.7/9.9 MB 3.4 MB/s eta 0:00:02
   ----------------------- ---------------- 5.8/9.9 MB 3.7 MB/s eta 0:00:02
   ----------------------------- ---------- 7.3/9.9 MB 4.1 MB/s eta 0:00:01
   ---------------------------------- ----- 8.7/9.9 MB 4.3 MB/s eta 0:00:01
   -------------------------------------

  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython-sql 0.4.1 requires prettytable<1, but you have prettytable 3.17.0 which is incompatible.


In [4]:
import os

folders = [
    "data/raw",
    "data/processed",
    "scripts",
    "dags",
    "dashboards",
    "docs",
    "airflow"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)
    print(f"✅ Created → {folder}")

print("\n📁 Project structure ready!")
print("""
fintech-behavior-analytics/
├── data/
│   ├── raw/
│   └── processed/
├── scripts/
├── dags/
├── dashboards/
├── docs/
├── airflow/
└── fintech_analytics.ipynb
""")

✅ Created → data/raw
✅ Created → data/processed
✅ Created → scripts
✅ Created → dags
✅ Created → dashboards
✅ Created → docs
✅ Created → airflow

📁 Project structure ready!

fintech-behavior-analytics/
├── data/
│   ├── raw/
│   └── processed/
├── scripts/
├── dags/
├── dashboards/
├── docs/
├── airflow/
└── fintech_analytics.ipynb



In [5]:
gitignore_content = """
.env
*.parquet
data/
airflow/logs/
airflow/db/
__pycache__/
*.pyc
""".strip()

with open(".gitignore", "w") as f:
    f.write(gitignore_content)

print("✅ .gitignore created!")
print("\nContents:")
print(gitignore_content)

✅ .gitignore created!

Contents:
.env
*.parquet
data/
airflow/logs/
airflow/db/
__pycache__/
*.pyc


In [6]:
requirements = """
pandas==2.1.0
boto3==1.28.0
python-dotenv==1.0.0
duckdb==0.10.0
pyarrow==14.0.1
apache-airflow==2.8.1
slack-sdk==3.27.1
""".strip()

with open("requirements.txt", "w") as f:
    f.write(requirements)

print("✅ requirements.txt created!")
print("\nContents:")
print(requirements)

✅ requirements.txt created!

Contents:
pandas==2.1.0
boto3==1.28.0
python-dotenv==1.0.0
duckdb==0.10.0
pyarrow==14.0.1
apache-airflow==2.8.1
slack-sdk==3.27.1


In [7]:
import duckdb

con = duckdb.connect()
result = con.execute("SELECT 'DuckDB is working!' AS message").fetchdf()
print(result)
print("\n✅ DuckDB hello world passed!")

              message
0  DuckDB is working!

✅ DuckDB hello world passed!


In [8]:
!git init

Initialized empty Git repository in C:/Users/Hi/Documents/Real-Time Customer Behavior Analytics Platform/fintech_analytics.ipynb/.git/


In [10]:
!git add .

In [11]:
!git commit -m "Initial project setup"

[main (root-commit) 8088b11] Initial project setup
 3 files changed, 14 insertions(+)
 create mode 100644 .gitignore
 create mode 100644 .ipynb
 create mode 100644 requirements.txt


In [12]:
# Replace YOUR_USERNAME with your actual GitHub username
# First create the repo on github.com, then run this

!git remote add origin https://github.com/mohimpact/fintech-behavior-analytics.git
!git push -u origin main

branch 'main' set up to track 'origin/main'.


To https://github.com/mohimpact/fintech-behavior-analytics.git
 * [new branch]      main -> main


In [13]:
import pandas as pd
import numpy as np
import random
import uuid
import os
import pyarrow as pa
import pyarrow.parquet as pq
from datetime import datetime, timedelta

# ── Reproducibility seeds ──────────────────────────────────────────────────
random.seed(42)
np.random.seed(42)

# ── Configuration ──────────────────────────────────────────────────────────
NUM_USERS   = 1000
NUM_EVENTS  = 50000
DAYS        = 90
START_DATE  = datetime(2024, 1, 1)
OUTPUT_DIR  = "data/raw"

NIGERIAN_CITIES = [
    "Lagos", "Abuja", "Port Harcourt", "Kano",
    "Ibadan", "Enugu", "Kaduna", "Benin City"
]

DEVICES        = ["android", "ios", "web"]
DEVICE_WEIGHTS = [0.55, 0.30, 0.15]    # Most Nigerians use Android

EVENT_TYPES = [
    "app_open", "login", "view_dashboard", "create_savings_plan",
    "deposit_money", "withdraw_money", "check_balance", "logout"
]

# Financial events with realistic Naira amount ranges
FINANCIAL_EVENTS = {
    "deposit_money":  (2000,  500000),
    "withdraw_money": (1000,  200000),
}

print("✅ Configuration loaded!")
print(f"   Users      : {NUM_USERS:,}")
print(f"   Events     : {NUM_EVENTS:,}")
print(f"   Date range : {START_DATE.date()} → {(START_DATE + timedelta(days=DAYS)).date()}")
print(f"   Cities     : {NIGERIAN_CITIES}")

✅ Configuration loaded!
   Users      : 1,000
   Events     : 50,000
   Date range : 2024-01-01 → 2024-03-31
   Cities     : ['Lagos', 'Abuja', 'Port Harcourt', 'Kano', 'Ibadan', 'Enugu', 'Kaduna', 'Benin City']


In [14]:
def create_user_profiles(num_users: int) -> pd.DataFrame:
    """
    Create user profiles with 3 realistic segments:
    - 70% lurkers  → low activity (open app rarely)
    - 20% regular  → weekly activity
    - 10% power    → daily activity
    """
    profiles = []

    for i in range(num_users):
        rand = random.random()

        if rand < 0.70:
            segment        = "lurker"
            events_per_day = np.random.uniform(0.05, 0.3)   # less than 1 event/day
        elif rand < 0.90:
            segment        = "regular"
            events_per_day = np.random.uniform(0.5, 2.0)    # a few times per week
        else:
            segment        = "power"
            events_per_day = np.random.uniform(3.0, 8.0)    # multiple times per day

        profiles.append({
            "user_id":        f"USR{i+1:05d}",
            "segment":        segment,
            "events_per_day": events_per_day,
            "city":           random.choice(NIGERIAN_CITIES),
            "device":         random.choices(DEVICES, weights=DEVICE_WEIGHTS)[0],
            "tenure_days":    random.randint(1, 365),
            "join_date":      START_DATE - timedelta(days=random.randint(1, 365)),
        })

    return pd.DataFrame(profiles)


# ── Run & preview ──────────────────────────────────────────────────────────
users_df = create_user_profiles(NUM_USERS)

print(f"✅ Created {len(users_df):,} user profiles\n")
print("Segment breakdown:")
print(users_df["segment"].value_counts())
print(f"\nSample profiles:")
users_df.head(5)

✅ Created 1,000 user profiles

Segment breakdown:
segment
lurker     697
regular    204
power       99
Name: count, dtype: int64

Sample profiles:


,user_id,segment,events_per_day,city,device,tenure_days,join_date
0,USR00001,lurker,0.143635,Lagos,ios,126,2023-09-08
1,USR00002,lurker,0.287679,Abuja,ios,280,2023-11-17
2,USR00003,lurker,0.232998,Lagos,android,112,2023-09-03
3,USR00004,lurker,0.199665,Lagos,ios,333,2023-01-06
4,USR00005,lurker,0.089005,Kano,android,143,2023-12-28


In [15]:
def realistic_timestamp(base_date: datetime) -> datetime:
    """
    Generate timestamps with realistic Nigerian time-of-day patterns.
    Peak hours: 8am (commute), 12pm (lunch), 6pm (evening)
    Weekends have ~30% less activity than weekdays.
    """
    hour_weights = [
        0.5, 0.3, 0.2, 0.2, 0.3, 0.5,    # 12am–5am  (very low)
        1.0, 1.5, 3.0, 2.5, 2.0, 2.5,    # 6am–11am  (morning peak at 8am)
        3.5, 2.5, 2.0, 2.0, 2.5, 3.5,    # 12pm–5pm  (lunch peak)
        4.0, 3.5, 2.5, 2.0, 1.5, 1.0,    # 6pm–11pm  (evening peak)
    ]

    hour   = random.choices(range(24), weights=hour_weights)[0]
    minute = random.randint(0, 59)
    second = random.randint(0, 59)

    ts = base_date.replace(hour=hour, minute=minute, second=second)

    # Weekend reduction — shift 30% of weekend events to weekdays
    if ts.weekday() >= 5:
        if random.random() < 0.30:
            ts = ts - timedelta(days=random.randint(1, 2))

    return ts


# ── Quick test ─────────────────────────────────────────────────────────────
test_ts = realistic_timestamp(datetime(2024, 1, 15))
print(f"✅ Timestamp generator working")
print(f"   Sample timestamp : {test_ts}")
print(f"   Day of week      : {test_ts.strftime('%A')}")
print(f"   Hour             : {test_ts.hour}:00")

✅ Timestamp generator working
   Sample timestamp : 2024-01-15 09:26:19
   Day of week      : Monday
   Hour             : 9:00


In [16]:
def build_event(user: dict, event_type: str, ts: datetime, session_id: str) -> dict:
    """Build a single event record with all required fields."""
    amount = None
    if event_type in FINANCIAL_EVENTS:
        lo, hi = FINANCIAL_EVENTS[event_type]
        amount = round(random.uniform(lo, hi), 2)

    return {
        "event_id":         str(uuid.uuid4()),
        "user_id":          user["user_id"],
        "event_type":       event_type,
        "timestamp":        ts,
        "amount":           amount,
        "session_id":       session_id,
        "device_type":      user["device"],
        "location_city":    user["city"],
        "user_tenure_days": user["tenure_days"],
        "user_segment":     user["segment"],
    }


def generate_session_events(user: dict, session_date: datetime) -> list:
    """
    Generate a realistic sequence of events for one user session.
    Flow: app_open → login → (actions) → logout
    """
    session_id = str(uuid.uuid4())[:8]
    events     = []
    ts         = realistic_timestamp(session_date)

    # Every session starts with app_open then login
    for event_type in ["app_open", "login"]:
        events.append(build_event(user, event_type, ts, session_id))
        ts += timedelta(seconds=random.randint(2, 10))

    # Number of middle actions depends on segment
    if user["segment"] == "power":
        num_actions = random.randint(2, 4)
    elif user["segment"] == "regular":
        num_actions = random.randint(1, 3)
    else:
        num_actions = random.randint(1, 2)

    # Middle actions with realistic weights
    mid_actions = [
        "view_dashboard", "check_balance",
        "create_savings_plan", "deposit_money", "withdraw_money"
    ]
    mid_weights = [0.35, 0.30, 0.15, 0.13, 0.07]

    for _ in range(num_actions):
        action = random.choices(mid_actions, weights=mid_weights)[0]
        events.append(build_event(user, action, ts, session_id))
        ts += timedelta(seconds=random.randint(15, 120))

    # 80% of sessions end with an explicit logout
    if random.random() < 0.80:
        events.append(build_event(user, "logout", ts, session_id))

    return events


print("✅ Event builder functions ready!")
print("   build_event()            → builds one event record")
print("   generate_session_events() → builds a full session flow")

✅ Event builder functions ready!
   build_event()            → builds one event record
   generate_session_events() → builds a full session flow


In [17]:
def generate_events(num_events: int = NUM_EVENTS) -> pd.DataFrame:
    """Generate all events across all users over the 90-day window."""

    print("Step 1: Creating user profiles...")
    users_df     = create_user_profiles(NUM_USERS)
    users        = users_df.to_dict("records")
    total_weight = users_df["events_per_day"].sum()

    print(f"Step 2: Generating ~{num_events:,} events over {DAYS} days...")
    all_events = []

    for user in users:
        # Calculate how many sessions this user gets over 90 days
        expected_sessions = int(
            (user["events_per_day"] / total_weight) * num_events / 3
        )
        expected_sessions = max(1, expected_sessions)

        # Randomly spread sessions across the 90-day window
        session_days = sorted(
            random.choices(range(DAYS), k=expected_sessions)
        )

        for day_offset in session_days:
            session_date = START_DATE + timedelta(days=day_offset)
            events       = generate_session_events(user, session_date)
            all_events.extend(events)

        # Stop once we hit our target
        if len(all_events) >= num_events:
            break

    # Build DataFrame, sort by time, trim to exact target
    df = pd.DataFrame(all_events)
    df = df.sort_values("timestamp").reset_index(drop=True)
    df = df.head(num_events)

    print(f"\n✅ Generated {len(df):,} events")
    print(f"   Date range    : {df['timestamp'].min().date()} → {df['timestamp'].max().date()}")
    print(f"   Unique users  : {df['user_id'].nunique():,}")
    print(f"   Unique sessions: {df['session_id'].nunique():,}")
    print(f"\n   Event type breakdown:")
    print(df["event_type"].value_counts().to_string())

    return df


# ── Run it ─────────────────────────────────────────────────────────────────
events_df = generate_events(NUM_EVENTS)
print(f"\nPreview of first 5 rows:")
events_df.head()

Step 1: Creating user profiles...
Step 2: Generating ~50,000 events over 90 days...

✅ Generated 50,000 events
   Date range    : 2024-01-01 → 2024-03-30
   Unique users  : 572
   Unique sessions: 9,293

   Event type breakdown:
event_type
app_open               9293
login                  9292
view_dashboard         8387
logout                 7420
check_balance          7138
create_savings_plan    3623
deposit_money          3203
withdraw_money         1644

Preview of first 5 rows:


,event_id,user_id,event_type,timestamp,amount,session_id,device_type,location_city,user_tenure_days,user_segment
0,713aa4ff-29f8-49b1-a35b-f6c691a89261,USR00339,app_open,2024-01-01 05:40:58,NaN,5e5f4a15,android,Port Harcourt,190,power
1,3a634597-8e1b-4792-8d1f-ffb85b524e34,USR00339,login,2024-01-01 05:41:04,NaN,5e5f4a15,android,Port Harcourt,190,power
2,928bc474-94e4-4da0-9a07-5ded0102e1fb,USR00339,deposit_money,2024-01-01 05:41:08,231685.7,5e5f4a15,android,Port Harcourt,190,power
3,bc4a7bdb-d7ea-4812-8257-a2941751c501,USR00339,check_balance,2024-01-01 05:41:52,NaN,5e5f4a15,android,Port Harcourt,190,power
4,85b76854-d69d-4718-b0c7-5b4237fcdee4,USR00339,view_dashboard,2024-01-01 05:42:26,NaN,5e5f4a15,android,Port Harcourt,190,power


In [18]:
def save_partitioned_parquet(df: pd.DataFrame, output_dir: str = OUTPUT_DIR):
    """
    Save events partitioned by year/month/day.
    Mirrors real production data lake structure:
    data/raw/year=2024/month=01/day=01/events.parquet
    """
    df = df.copy()
    df["year"]  = df["timestamp"].dt.year.astype(str)
    df["month"] = df["timestamp"].dt.month.apply(lambda x: f"{x:02d}")
    df["day"]   = df["timestamp"].dt.day.apply(lambda x: f"{x:02d}")

    groups = df.groupby(["year", "month", "day"])
    print(f"Saving {len(groups)} daily partition files...")

    for (year, month, day), group in groups:
        partition_path = os.path.join(
            output_dir,
            f"year={year}",
            f"month={month}",
            f"day={day}"
        )
        os.makedirs(partition_path, exist_ok=True)

        file_path = os.path.join(partition_path, "events.parquet")
        group.drop(columns=["year", "month", "day"]).to_parquet(
            file_path, index=False
        )

    print(f"\n✅ Partitioned files saved to {output_dir}/")
    print(f"   Total daily partitions : {len(groups)}")
    print(f"   Example path           : {output_dir}/year=2024/month=01/day=01/events.parquet")


# ── Run it ─────────────────────────────────────────────────────────────────
os.makedirs(OUTPUT_DIR, exist_ok=True)
save_partitioned_parquet(events_df)

Saving 90 daily partition files...

✅ Partitioned files saved to data/raw/
   Total daily partitions : 90
   Example path           : data/raw/year=2024/month=01/day=01/events.parquet


In [19]:
def save_flat_parquet(df: pd.DataFrame):
    """
    Save a single flat parquet file.
    This is what we'll load into Power BI later.
    """
    flat_path = "data/processed/events_flat.parquet"
    os.makedirs("data/processed", exist_ok=True)
    df.to_parquet(flat_path, index=False)

    size_kb = os.path.getsize(flat_path) / 1024
    print(f"✅ Flat file saved → {flat_path}")
    print(f"   Rows  : {len(df):,}")
    print(f"   Cols  : {len(df.columns)}")
    print(f"   Size  : {size_kb:.1f} KB")


# ── Run it ─────────────────────────────────────────────────────────────────
save_flat_parquet(events_df)

✅ Flat file saved → data/processed/events_flat.parquet
   Rows  : 50,000
   Cols  : 10
   Size  : 2552.2 KB


In [20]:
import duckdb

con = duckdb.connect()

print("=== Event Type Summary ===")
event_summary = con.execute("""
    SELECT
        event_type,
        COUNT(*)                    AS total_events,
        COUNT(DISTINCT user_id)     AS unique_users,
        COUNT(DISTINCT session_id)  AS unique_sessions,
        ROUND(AVG(amount), 2)       AS avg_amount_naira
    FROM read_parquet('data/raw/**/*.parquet')
    GROUP BY event_type
    ORDER BY total_events DESC
""").fetchdf()

event_summary

=== Event Type Summary ===


,event_type,total_events,unique_users,unique_sessions,avg_amount_naira
0,app_open,9293,572,9293,NaN
1,login,9292,572,9292,NaN
2,view_dashboard,8387,461,5985,NaN
3,logout,7420,548,7420,NaN
4,check_balance,7138,451,5336,NaN
5,create_savings_plan,3623,351,3140,NaN
6,deposit_money,3203,335,2790,253966.99
7,withdraw_money,1644,267,1536,101471.29


In [21]:
print("=== User Segment Breakdown ===")
segment_summary = con.execute("""
    SELECT
        user_segment,
        COUNT(DISTINCT user_id)    AS users,
        COUNT(*)                   AS total_events,
        COUNT(DISTINCT session_id) AS total_sessions
    FROM read_parquet('data/raw/**/*.parquet')
    GROUP BY user_segment
    ORDER BY users DESC
""").fetchdf()

segment_summary

=== User Segment Breakdown ===


,user_segment,users,total_events,total_sessions
0,lurker,381,4076,953
1,regular,124,12043,2500
2,power,67,33881,5840


In [22]:
print("=== Hourly Activity Pattern ===")
hourly = con.execute("""
    SELECT
        EXTRACT(HOUR FROM timestamp) AS hour,
        COUNT(*)                     AS events
    FROM read_parquet('data/raw/**/*.parquet')
    GROUP BY hour
    ORDER BY hour
""").fetchdf()

hourly

=== Hourly Activity Pattern ===


,hour,events
0,0,621
1,1,243
2,2,196
3,3,158
4,4,302
5,5,466
6,6,1178
7,7,1742
8,8,3035
9,9,2494


In [23]:
print("=== Final Dataset Overview ===\n")
print(f"Shape          : {events_df.shape}")
print(f"Columns        : {list(events_df.columns)}")
print(f"Date range     : {events_df['timestamp'].min().date()} → {events_df['timestamp'].max().date()}")
print(f"Unique users   : {events_df['user_id'].nunique():,}")
print(f"Unique sessions: {events_df['session_id'].nunique():,}")
print(f"\nNull values per column:")
print(events_df.isnull().sum())
print(f"\nRandom sample of 5 rows:")
events_df.sample(5)

=== Final Dataset Overview ===

Shape          : (50000, 10)
Columns        : ['event_id', 'user_id', 'event_type', 'timestamp', 'amount', 'session_id', 'device_type', 'location_city', 'user_tenure_days', 'user_segment']
Date range     : 2024-01-01 → 2024-03-30
Unique users   : 572
Unique sessions: 9,293

Null values per column:
event_id                0
user_id                 0
event_type              0
timestamp               0
amount              45153
session_id              0
device_type             0
location_city           0
user_tenure_days        0
user_segment            0
dtype: int64

Random sample of 5 rows:


,event_id,user_id,event_type,timestamp,amount,session_id,device_type,location_city,user_tenure_days,user_segment
38783,cc22b6d1-12f7-479c-8904-0998e00e71e7,USR00528,check_balance,2024-03-10 17:21:43,NaN,9ba922e1,android,Benin City,66,power
32961,e1b11410-6d2b-4e51-a1de-4d350e17a55d,USR00256,create_savings_plan,2024-02-29 15:46:39,NaN,db20ac73,android,Benin City,275,power
25636,4419d726-a6eb-447d-9173-80a8214fd1ab,USR00325,check_balance,2024-02-15 23:22:28,NaN,c1d01b1f,android,Abuja,235,power
5683,43bee29f-d8a8-4c17-97e1-9d3f6c3b51e0,USR00150,deposit_money,2024-01-10 22:29:52,357486.51,bd8ae607,ios,Enugu,10,lurker
19242,0d9d1da2-390a-44ba-bfff-4b5063919b87,USR00116,check_balance,2024-02-04 14:24:01,NaN,9752aeb9,ios,Port Harcourt,64,power


In [24]:
!git add .

In [25]:
!git commit -m "Days 1-4: Event generator — 50K fintech events, partitioned parquet"

On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean


In [26]:
!git push origin main

Everything up-to-date


In [27]:
!git config --global core.autocrlf true